# SDXL vs FreeU vs PAG — Colab 运行笔记本

准备：把整个 `sdxl-freeu-pag` 文件夹上传到 Google Drive 的 `MyDrive/` 下。
运行时类型选 **GPU（A100 或 L4 最好）**。所有输出直接写在 Drive 里，断线不会丢。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/sdxl-freeu-pag
!nvidia-smi

In [ ]:
!pip install -q -r requirements.txt

## 1. 抽样 prompt（只需运行一次，所有人共用同一份 data/prompts.csv）

In [ ]:
!python scripts/01_sample_prompts.py --n 150 --subset 50
!head -5 data/prompts.csv

## 2. 冒烟测试：每组 5 张

In [ ]:
!python scripts/02_generate.py --group main --limit 5

In [ ]:
from PIL import Image
from IPython.display import display
import glob
for run in ['baseline', 'freeu', 'pag']:
    f = sorted(glob.glob(f'outputs/images/{run}/*.png'))[0]
    print(run, f); display(Image.open(f).resize((384, 384)))

## 3. 正式生成（断线后重新运行同一格即可续跑）

In [ ]:
!python scripts/02_generate.py --group main

In [ ]:
!python scripts/02_generate.py --group freeu_sweep --group pag_sweep

In [ ]:
# 可选：PAG 作用层 + FreeU/PAG 叠加
!python scripts/02_generate.py --runs pag_up freeu_pag

## 4. CLIPScore

In [ ]:
!python scripts/03_clipscore.py

## 5. 打包盲评
运行后下载 `blind_eval/` 文件夹（或在 Drive 里直接分享），三人各自打开 `index.html` 评分，
导出的 `results_<名字>.json` 上传到 Drive 的 `sdxl-freeu-pag/eval_results/`。

In [ ]:
!python scripts/04_build_blind_eval.py
!cd /content/drive/MyDrive/sdxl-freeu-pag && zip -qr blind_eval.zip blind_eval && ls -lh blind_eval.zip

## 6. 分析与作图（盲评结果收回后再运行一次）

In [ ]:
!python scripts/05_analyze.py
!python scripts/06_make_grids.py

In [ ]:
from IPython.display import Markdown, Image as IImage
display(Markdown(open('results/summary.md').read()))
for f in sorted(glob.glob('results/fig_*.png')):
    print(f); display(IImage(f, width=520))